# 05 — DPO ile Alignment (Hizalama)

**Kapsam:** RLHF, DPO, PPO veya benzeri model hizalama (alignment) yöntemleri.

## RLHF/PPO vs. DPO
Klasik RLHF üç aşamalıdır: (1) SFT, (2) ayrı bir **ödül modeli** eğitmek, (3) bu ödül
modelini kullanarak **PPO** ile politika modelini güncellemek. Bu, iki model + kararsız
bir RL döngüsü gerektirir.

**DPO (Direct Preference Optimization)**, tercih çiftlerini (chosen/rejected) doğrudan
bir kayıp fonksiyonuna çevirir — ayrı ödül modeli veya RL rollout'u gerekmez. Matematiksel
olarak PPO'nun optimum noktasına eşdeğer bir çözüme ulaşır, ama çok daha kararlı ve
ucuzdur. Bu yüzden küçük ölçekli projelerde tercih ediyoruz.

Bu notebook, 04. notebook'ta ürettiğiniz QLoRA adaptörünün üzerine devam eder.

In [ ]:
# Bu hücre HER notebook'ta ayrı ayrı çalıştırılmalı: Colab'da her sekme/notebook
# genellikle kendi çalışma zamanını (VM) alır, yani /content her seferinde sıfırdanmış
# gibi başlar. Bu hücre kendi kendini onaran bir kurulum yapar:
#   1) Proje klasörü zaten varsa (aynı çalışma zamanında önceki hücre/notebook
#      tarafından kurulmuşsa) unpack adımını atlar.
#   2) Yoksa Google Drive'ı mount edip, DRIVE_ZIP_PATH'teki zip'i /content'e açar
#      (zip'in içinde 'baykar-nlp-hazirlik/' klasörü kök olarak yer almalı).
#   3) Drive'da zip de yoksa, kendi GitHub reponuzu klonlamanız için bir uyarı basar.
#   4) EN ÖNEMLİSİ: data/, models/, mlruns/ klasörlerini Drive'daki kalıcı bir
#      klasöre sembolik bağlantı (symlink) yapar. Neden gerekli: /content her
#      runtime'da sıfırlanır, yani 01. notebook'ta ürettiğiniz corpus.jsonl gibi
#      dosyalar farklı bir runtime'da (örn. 02. notebook'u açtığınızda) KAYBOLUR.
#      Bu adım olmadan her notebook'u ayrı ayrı çalıştırdığınızda önceki adımların
#      ürettiği veriyi bulamazsınız. Sembolik bağlantı sayesinde hangi runtime'da
#      olursanız olun aynı kalıcı depoyu okur/yazarsınız.
import os, sys
os.environ.setdefault("USE_TF", "0")  # transformers TensorFlow'u hic denemesin (Colab'da protobuf catismasi yasatiyor)

PROJECT_DIR = "/content/baykar-nlp-hazirlik"
DRIVE_ZIP_PATH = "/content/drive/MyDrive/baykar-nlp-hazirlik.zip"
DRIVE_DATA_DIR = "/content/drive/MyDrive/baykar-nlp-hazirlik-data"
PERSIST_DIRS = ["data/raw", "data/processed", "models", "mlruns"]  # chroma_db BILEREK haric (asagida)

try:
    from google.colab import drive
    # force_remount=True KULLANMIYORUZ: bu, zaten mount edilmişken bile her seferinde
    # yeniden yetkilendirme (izin penceresi) ister, gereksiz bekleme/kesinti yaratır.
    # drive.mount() zaten mount edilmişse kendi içinde anında geri döner; mount
    # edilmemişse (bu runtime'da ilk çalıştırma) normal şekilde izin ister — bu
    # durumda çıkan izin penceresini/bağlantısını tamamlamanız gerekir, hücreyi
    # durdurmayın.
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False  # Colab dışında (yerelde) çalışıyorsanız Drive adımları atlanır.

if not os.path.exists(PROJECT_DIR) and IN_COLAB:
    if os.path.exists(DRIVE_ZIP_PATH):
        import shutil
        shutil.unpack_archive(DRIVE_ZIP_PATH, "/content")
    else:
        print(f"UYARI: {DRIVE_ZIP_PATH} bulunamadı. Zip'i Drive'ınızın köküne "
              "yükleyin ya da kendi reponuzu klonlayın: "
              f"!git clone <repo-url> {PROJECT_DIR}")

if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

if IN_COLAB and os.path.exists(PROJECT_DIR):
    import shutil
    os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
    for _name in PERSIST_DIRS:
        _drive_path = os.path.join(DRIVE_DATA_DIR, _name)
        os.makedirs(_drive_path, exist_ok=True)
        _local_path = os.path.join(PROJECT_DIR, _name)
        os.makedirs(os.path.dirname(_local_path), exist_ok=True)  # orn. data/ klasorunu gercek dizin olarak olustur

        if os.path.islink(_local_path):
            continue  # zaten Drive'a bağlanmış

        if os.path.isdir(_local_path):
            # Zip'ten gelen boş klasörü kaldırıp yerine symlink koyuyoruz. İçinde
            # (nadiren) veri varsa önce Drive'a taşıyoruz, hiçbir şeyi kaybetmiyoruz.
            for _item in os.listdir(_local_path):
                _src = os.path.join(_local_path, _item)
                _dst = os.path.join(_drive_path, _item)
                if not os.path.exists(_dst):
                    shutil.move(_src, _dst)
            shutil.rmtree(_local_path)

        os.symlink(_drive_path, _local_path)

    print("Kalıcı veri klasörü:", DRIVE_DATA_DIR)
    print("Not: chroma_db (vektor veritabani) Drive'a BAGLANMADI -- SQLite, Drive'in")
    print("     FUSE dosya sisteminde yazma kilidini desteklemiyor ('OperationalError:")
    print("     attempt to write a readonly database'). Her yeni runtime'da RAG")
    print("     notebook'undaki (03) indeksleme hucresini tekrar calistirin -- chunks.jsonl")
    print("     zaten Drive'da oldugu icin bu hizli ve ucretsiz bir islemdir.")


## Colab ortam düzeltmeleri

Kurulum hücresinden hemen sonra çalıştırın. Paket sürümlerini sabitler. Karşılaştırma
hücresi RAG kullanır → `chromadb` gerekir.

1. **Kurulum hücresini** çalıştırın
2. **Runtime → Oturumu yeniden başlat**
3. **Doğrulama hücresini** çalıştırın

In [ ]:
import os, subprocess, sys

PROJECT_DIR = "/content/baykar-nlp-hazirlik"
bootstrap = os.path.join(PROJECT_DIR, "scripts", "colab_bootstrap.py")

if not os.path.exists(bootstrap):
    raise FileNotFoundError(
        "scripts/colab_bootstrap.py bulunamadi. "
        "Guncel projeyi zip'leyip Drive'a yukleyin."
    )

subprocess.run([sys.executable, bootstrap], check=True, cwd=PROJECT_DIR)

### Doğrulama (restart sonrası)

In [ ]:
import os, sys
PROJECT_DIR = "/content/baykar-nlp-hazirlik"
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

import transformers, chromadb, trl, pyarrow, datasets
import huggingface_hub
from numpy._core import strings  # eskiden "_center" ImportError'i buradan geliyordu
from transformers.models.qwen2 import modeling_qwen2  # bu projenin gercek modeli -- scipy/numpy ABI sorunlari (orn. "_blas_supports_fpe") transformers'in object-detection loss modulu uzerinden burada ortaya cikardi
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("chromadb:", chromadb.__version__)
print("trl:", trl.__version__)
print("pyarrow:", pyarrow.__version__)
print("datasets:", datasets.__version__)
print("OK -- devam edebilirsiniz.")

## 1. Tercih (preference) veri seti üretimi

'Chosen' = iyi yapılandırılmış hedef doküman, 'rejected' = şablon bağlamı verilmeden üretilen serbest/tutarsız cevap.

In [ ]:
from src.alignment.preference_dataset import build_preference_dataset, save_preference_dataset

pairs = build_preference_dataset(max_examples=150)
path = save_preference_dataset(pairs)
print(f"{len(pairs)} tercih çifti kaydedildi -> {path}")
print("\nÖrnek:")
print("Chosen:", pairs[0]["chosen"][:200], "...")
print("Rejected:", pairs[0]["rejected"][:200], "...")


## 2. DPO eğitimi

In [ ]:
from src.alignment.dpo_train import train
from src.config import QLORA_CONFIG

dpo_adapter_path = train(sft_adapter_dir=QLORA_CONFIG.output_dir)
print("DPO adaptörü kaydedildi ->", dpo_adapter_path)


## 3. Karşılaştırma

Aynı taslak notları SFT-only ve SFT+DPO modelleriyle karşılaştırın; DPO sonrası
dokümanların daha tutarlı ve daha az 'uydurma ayrıntı içeren' olmasını bekleriz.

In [ ]:
import json
from src.rag.vector_store import get_collection, index_chunks

# chroma_db Drive'a baglanmiyor, yani bu runtime'da 03. notebook'un indeksleme adimi
# hic calismadiysa koleksiyon bos olur ve asagidaki answer() cagrilari "Bu bilgi
# elimdeki dokumanlarda yok." fallback'ini doner. Bos ise burada kendi kendine
# yeniden indeksliyoruz.
if get_collection().count() == 0:
    print("Chroma indeksi bu runtime'da bos -- yeniden indeksleniyor...")
    with open("data/processed/chunks.jsonl", encoding="utf-8") as f:
        chunks = [json.loads(line) for line in f]
    index_chunks(chunks)
    print(f"{len(chunks)} chunk indekslendi.")

In [ ]:
import os
from src.config import QLORA_CONFIG, MODELS_DIR
from src.rag.rag_pipeline import answer

# QLORA_CONFIG ve dpo_adapter_path bu notebook'un "2. DPO egitimi" hucresinde
# tanimlaniyordu; runtime restart sonrasi ya da notebook'u dogrudan bu hucreden
# baslatirsaniz bellekte olmazlar. Ikisi de Drive'a kalici oldugu icin (models/
# symlink'li) yeniden egitmeye gerek yok -- dogrudan bilinen diskteki yollarina
# isaret ediyoruz.
dpo_adapter_path = str(MODELS_DIR / "dpo-adapter")
for _label, _path in [("QLoRA", QLORA_CONFIG.output_dir), ("DPO", dpo_adapter_path)]:
    if not os.path.exists(_path):
        raise FileNotFoundError(
            f"{_label} adaptoru bulunamadi: {_path} -- once ilgili egitim hucresini calistirin."
        )

q = "Aşağıdaki kaba taslak notları iyi yapılandırılmış bir SSS bölümüne dönüştür.\n\nTaslak notlar:\n- şifre sıfırlama nasıl\n- veri saklama süresi ne kadar"
print("SFT-only:\n", answer(q, model_path=QLORA_CONFIG.output_dir)["answer"])
print("\nSFT+DPO:\n", answer(q, model_path=dpo_adapter_path)["answer"])